# Einstein Summation

$$\begin{equation}
\begin{split}
\left( A \cdot B \right)_{i,k} &= \sum_{j}\left( A_{i,j} \cdot B_{j,k} \right) \\
\left( A \cdot B \right)_{i,k} &= A_{i,j} \cdot B_{j,k}
\end{split}
\end{equation}$$

`einsum('ij,jk->ik')` # sum over j index


In [ ]:
import torch

a = torch.tensor([1,2,3]) # tensor with shape [3], so i ranges from 0 to 2
b = torch.tensor([4,5,6]) # tensor with shape [3], so j ranges from 0 to 2

# Outer product of a (3x1) and b (3x1)
# a⊗b = ab^T
# [[1 2 3]] [[4 5 6]   [4   5  6]
#            [4 5 6] = [8  10 12]
#            [4 5 6]]  [12 15 18]
# 31,13->33
outer_product = torch.outer(a,b)
outer_product_einsum = torch.einsum('i,j->ij', a, b)

M = torch.empty((3,3))
for i in range(3):
  for j in range(3):
    M[i,j] = a[i]*b[j]

print("Outer product")
print(outer_product)
print(outer_product_einsum)
print(M)

Outer product
tensor([[ 4,  5,  6],
        [ 8, 10, 12],
        [12, 15, 18]])
tensor([[ 4,  5,  6],
        [ 8, 10, 12],
        [12, 15, 18]])
tensor([[ 4.,  5.,  6.],
        [ 8., 10., 12.],
        [12., 15., 18.]])


In [123]:
# Dot product
# [1 2 3] [4] = 32
#         [5]
#         [6]
dot_product = torch.dot(a,b)
dot_product_einsum = torch.einsum('i,i->', a, b) # 'i,i->sums over the index i'

print("Dot product")
print(dot_product)
print(dot_product_einsum)

Dot product
tensor(32)
tensor(32)


In [ ]:
# Matmul
# [[1 2 3]  [[7  8]     [[58 64]
#  [4 5 6]]  [9 10]   =  [139 154]
#            [11 12]]
matrix_a = torch.tensor([[1,2,3],
                         [4,5,6]])
matrix_b = torch.tensor([[7,  8],
                         [9, 10],
                         [11,12]])
matmul = torch.matmul(matrix_a, matrix_b)
matmul_einsum = torch.einsum('ij,jk->ik', matrix_a, matrix_b) # $$\sum_j A_{ij} \cdot B_{jk}$$
matmul_einsum = torch.einsum('ik,kj->ij', matrix_a, matrix_b) # $$\sum_k A_{ik} \cdot B_{kj}$$ same thing as above
M = torch.empty((2,2))
for i in range(2):
  for j in range(2):
    for k in range(3):
      M[i,j] += matrix_a[i,k] * matrix_b[k,j]

print("Matrix multiplication")
print(matmul)
print(matmul_einsum)
print(M)

Matrix multiplication
tensor([[ 58,  64],
        [139, 154]])
tensor([[ 58,  64],
        [139, 154]])
tensor([[ 58.,  64.],
        [139., 154.]])


In [260]:
w = torch.tensor([[0, 1, 2],
                  [3, 4, 5]]) # 2x3
x = torch.tensor([[1, 1, 2]]) # 1x3

# Elementwise multiplication
# Einsum: row-column
# [[0 1 2]   [[1 1 2]    [[0 1  4]
#  [3 4 5]]   [1 1 2]] =  [3 4 10]]
elementwise = w * x # PyTorch broadcasts `x` to match the shape of `w`
elementwise_einsum = torch.einsum('ij,kj->ij', w, x)
print("Elementwise multiplication")
print(elementwise)
print(elementwise_einsum)

Elementwise multiplication
tensor([[ 0,  1,  4],
        [ 3,  4, 10]])
tensor([[ 0,  1,  4],
        [ 3,  4, 10]])


In [258]:
# Matrix multiplication
# Einsum: row-row
# [[0 1 2]          = [[5]
#  [3 4 5]] * [[1]     [17]]
#              [1]
#              [2]]
# A_{i,j} \cdot B_{k,j}; summing over the index j
matmul = torch.matmul(w, x.transpose(0, 1))
matmul = w @ x.transpose(0, 1) # Equivalent to the line above
matmul_einsum = torch.einsum('ij,kj->ik', w, x) # Equivalent to the line above
print("Matrix multiplication")
print(matmul)
print(matmul_einsum)

Matrix multiplication
tensor([[ 5],
        [17]])
tensor([[ 5],
        [17]])


In [285]:
a = torch.tensor([[1,2,3],
                  [4,5,6]])
b = torch.tensor([[7,8,9],
                  [10,11,12]])
torch.matmul(a,b.transpose(0,1)) # 2x3 * 3x2 = 2x2

tensor([[ 50,  68],
        [122, 167]])

In [ ]:
# Matrix multiplication
a = torch.tensor([[1,2,3,4],
                  [5,6,7,8],
                  [9,10,11,12]]) # 3x4
b = torch.tensor([[1,2,3,4,5],
                  [6,7,8,9,10],
                  [11,12,13,14,15],
                  [16,17,18,19,20]]) # 4x5
matmul = torch.matmul(a,b)
matmul_einsum = torch.einsum("ij,jk->ik", a, b)
M = torch.empty((3,5))
for i in range(3):
  for k in range(5):
    for j in range(4):
      M[i,k] += a[i,j] * b[j,k]
print(matmul)
print(matmul_einsum)
print(M)

tensor([[110, 120, 130, 140, 150],
        [246, 272, 298, 324, 350],
        [382, 424, 466, 508, 550]])
tensor([[110, 120, 130, 140, 150],
        [246, 272, 298, 324, 350],
        [382, 424, 466, 508, 550]])
tensor([[110., 120., 130., 140., 150.],
        [246., 272., 298., 324., 350.],
        [382., 424., 466., 508., 550.]])


In [312]:
# Batch matrix multiplication
a = torch.tensor([[[1,2,3,4],
                   [5,6,7,8],
                   [9,10,11,12]],
                  [[12,13,14,15],
                   [16,17,18,19],
                   [20,21,22,23]]]) # 2x3x4
b = torch.tensor([[[1,2,3,4,5],
                   [6,7,8,9,10],
                   [11,12,13,14,15],
                   [16,17,18,19,20]],
                  [[21,22,23,24,25],
                   [26,27,28,29,30],
                   [31,32,33,34,35],
                   [36,37,38,39,40]]]) # 2x4x5
matmul = torch.matmul(a, b)
matmul_einsum = torch.einsum("ijk,ikl->ijl", a, b)
M = torch.zeros((2,3,5))
for i in range(2):
  for j in range(3):
    for l in range(5):
      for k in range(4):
        M[i,j,l] += a[i,j,k] * b[i,k,l]
print(matmul)
print(matmul_einsum)
print(M)

tensor([[[ 110,  120,  130,  140,  150],
         [ 246,  272,  298,  324,  350],
         [ 382,  424,  466,  508,  550]],

        [[1564, 1618, 1672, 1726, 1780],
         [2020, 2090, 2160, 2230, 2300],
         [2476, 2562, 2648, 2734, 2820]]])
tensor([[[ 110,  120,  130,  140,  150],
         [ 246,  272,  298,  324,  350],
         [ 382,  424,  466,  508,  550]],

        [[1564, 1618, 1672, 1726, 1780],
         [2020, 2090, 2160, 2230, 2300],
         [2476, 2562, 2648, 2734, 2820]]])
tensor([[[ 110.,  120.,  130.,  140.,  150.],
         [ 246.,  272.,  298.,  324.,  350.],
         [ 382.,  424.,  466.,  508.,  550.]],

        [[1564., 1618., 1672., 1726., 1780.],
         [2020., 2090., 2160., 2230., 2300.],
         [2476., 2562., 2648., 2734., 2820.]]])


In [317]:
# Matrix Dialonal
x = torch.tensor([[1,2,3],
                  [4,5,6],
                  [7,8,9]])
diag = torch.diag(x)
diag_einsum = torch.einsum("ii->i", x)
M = torch.empty(3)
for i in range(3):
  M[i] = x[i][i]
print(diag)
print(diag_einsum)
print(M)

tensor([1, 5, 9])
tensor([1, 5, 9])
tensor([1., 5., 9.])


In [321]:
# Matrix Trace (sum of the elements of the diagonal of the input 2-D matrix)
x = torch.tensor([[1,2,3],
                  [4,5,6],
                  [7,8,9]])
trace = torch.trace(x)
trace_einsum = torch.einsum("ii->", x)
M = torch.tensor(0)
for i in range(3):
  M += x[i][i]
print(trace)
print(trace_einsum)
print(M)

tensor(15)
tensor(15)
tensor(15)
